In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🏅 Medallion Architecture Mock — Bronze / Silver / Gold
# MAGIC
# MAGIC Gera **5 milhões de linhas** de dados de vendas (e-commerce) e processa
# MAGIC em três camadas usando **PySpark + Delta Lake**.
# MAGIC
# MAGIC - **Bronze** → dados crus, "sujos" (nulos, duplicados, tipos errados, datas em texto)
# MAGIC - **Silver** → dados limpos, tipados, deduplicados e validados
# MAGIC - **Gold** → tabelas agregadas prontas para BI / análise
# MAGIC
# MAGIC Feito para rodar no **Databricks Free Edition** (serverless + Unity Catalog).

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Configuração — catálogo e schema

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql import types as T

# No Free Edition o catálogo padrão é "workspace".
CATALOG = "workspace"
SCHEMA  = "medallion_demo"
N_ROWS  = 5_000_000   # 5 milhões

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Usando {CATALOG}.{SCHEMA} | gerando {N_ROWS:,} linhas")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. 🥉 BRONZE — geração de dados crus (distribuída, rápida)
# MAGIC
# MAGIC Usamos `spark.range()` para gerar as linhas em paralelo nos workers
# MAGIC (muito mais rápido que loop em Python). Injetamos sujeira de propósito:
# MAGIC categorias em caixa misturada, nulos, preços inválidos, datas em texto e duplicatas.

# COMMAND ----------

# Listas de domínio
categorias = ["Eletronicos", "Roupas", "Alimentos", "Livros",
              "Casa", "Brinquedos", "Esportes", "Beleza"]
paises     = ["BR", "US", "AR", "CL", "MX", "PT"]
pagamentos = ["credit_card", "debit_card", "pix", "boleto", "paypal"]

def pick(col_rand, valores):
    """Seleciona um elemento aleatório de uma lista via índice."""
    arr = F.array(*[F.lit(v) for v in valores])
    idx = (F.floor(col_rand * F.lit(len(valores))) + 1).cast("int")
    return F.element_at(arr, idx)

base = spark.range(N_ROWS).withColumnRenamed("id", "row_id")

bronze_df = (
    base
    .withColumn("transaction_id", F.concat(F.lit("TX-"), F.col("row_id").cast("string")))
    .withColumn("customer_id", (F.floor(F.rand(seed=11) * 200_000) + 1).cast("int"))
    .withColumn("product_id",  (F.floor(F.rand(seed=22) * 5_000) + 1).cast("int"))
    # categoria com caixa inconsistente (sujeira proposital)
    .withColumn("_cat", pick(F.rand(seed=33), categorias))
    .withColumn("category",
        F.when(F.rand(seed=34) < 0.33, F.upper(F.col("_cat")))
         .when(F.rand(seed=34) < 0.66, F.lower(F.col("_cat")))
         .otherwise(F.col("_cat")))
    .withColumn("country", pick(F.rand(seed=44), paises))
    .withColumn("payment_method", pick(F.rand(seed=55), pagamentos))
    # preço: ~3% inválido (zero ou negativo)
    .withColumn("unit_price",
        F.when(F.rand(seed=66) < 0.03, F.round(F.rand(seed=67) * -50, 2))
         .otherwise(F.round(F.rand(seed=68) * 990 + 10, 2)))
    .withColumn("quantity", (F.floor(F.rand(seed=77) * 5) + 1).cast("int"))
    # data como STRING (formato cru, como costuma chegar de origem)
    .withColumn("event_ts",
        F.from_unixtime(
            F.lit(1704067200) + (F.rand(seed=88) * 31_536_000).cast("long")
        ))  # vira string "yyyy-MM-dd HH:mm:ss"
    # ~2% de e-mails nulos
    .withColumn("email",
        F.when(F.rand(seed=99) < 0.02, F.lit(None))
         .otherwise(F.concat(F.lit("user"), F.col("customer_id").cast("string"), F.lit("@mail.com"))))
    .withColumn("_ingest_ts", F.current_timestamp())
    .drop("_cat", "row_id")
)

# Injeta ~1% de DUPLICATAS reais (union de uma amostra)
dups = bronze_df.sample(fraction=0.01, seed=123)
bronze_df = bronze_df.unionByName(dups)

(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_sales"))

#print("Bronze gravada:", spark.table("bronze_sales").count(), "linhas")
#display(spark.table("bronze_sales").limit(10))
